# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WingManOO7/ML-INTER/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Lane: Refresh / Content Opportunity Scoring (Lane 2)**

**Primary ML Task Type: Priority Ranking / Scoring** (supported by calibrated probabilistic **Binary Classification**).

### Why Ranking / Priority Scoring?
In real search and content operations, the core operational decision is never an unconstrained prediction in a vacuum. It is a triage and resource allocation decision: *"Which content pages out of thousands should an editorial or SEO team review and refresh first during this planning cycle?"*

- **Why not unconstrained regression?** We do not need to predict exact future raw impression counts or continuous traffic volume with high numerical precision. A page losing 1,200 impressions vs 1,400 impressions has the same operational urgency; predicting the exact continuous count introduces unnecessary variance without improving the triage decision.
- **Why not unsupervised clustering alone?** Clustering (Lane 3) groups pages into archetypes (e.g. "stale champions" or "decaying striking-distance"), which provides valuable context, but it does not order candidates from #1 to #50 for an editor's weekly sprint.
- **Why not pure binary classification?** In our dataset, 16,262 out of 30,000 pages (54.2%) have a negative trend direction. If we treated this solely as a binary classification problem with a 0.5 threshold, the model would classify over 16,000 pages as "declining." An editorial team with capacity to rewrite or refresh 20 to 50 pages per week cannot act on 16,000 positive flags. Pure classification leaves the operational triage problem completely unsolved.

Therefore, the task is properly framed as **Priority Ranking / Scoring**: we train an interpretable machine learning model to estimate the probability of decline/opportunity $\hat{p} \in [0, 1]$, and combine it with observable impact signals (such as 90-day search visibility and position tier) to generate an ordered queue of actionable candidates.

In [1]:
# Code check: Demonstrate why binary classification alone fails operational capacity
import os
import pandas as pd
import numpy as np

# Locate data path safely across environments
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
DATA_PATH = os.path.join(REPO_ROOT, 'data', 'raw', 'content_refresh_anonymized.csv')
df = pd.read_csv(DATA_PATH)

total_pages = len(df)
declining_pages = (df['trend_direction'] == 'down').sum()
base_rate = declining_pages / total_pages

print(f"Total inventory: {total_pages:,} pages")
print(f"Binary positive cases (trend_direction == 'down'): {declining_pages:,} ({base_rate:.1%})")
print(f"Editorial capacity: typically 20-50 pages per sprint")
print(f"Ratio of positive cases to weekly capacity: {declining_pages / 50:.0f}x")
print("Conclusion: Pure binary classification leaves 16,000+ positive cases unprioritized. Ranking is mandatory.")


Total inventory: 30,000 pages
Binary positive cases (trend_direction == 'down'): 16,262 (54.2%)
Editorial capacity: typically 20-50 pages per sprint
Ratio of positive cases to weekly capacity: 325x
Conclusion: Pure binary classification leaves 16,000+ positive cases unprioritized. Ranking is mandatory.


## 2. Target or proxy

### Starter Dataset Target: A Measured Proxy Label
In the starter dataset, the target is defined as:
$$\text{is\_declining\_label} = (\text{trend\_direction} == \text{"down"})$$

- **Where does it come from?** In the starter data, `trend_direction` is derived by comparing impressions in the trailing 30 days (`impressions_last_30d`) against the prior 30 days (`impressions_prev_30d`). A drop of greater than 20% is assigned `"down"`.
- **Is it an observed outcome or a defined rule?** It is an **observed outcome** measured from search logs (not a human's arbitrary quality tag), but in this starter slice it functions as a **proxy label** because it is computed over a recent comparison window within the snapshot rather than an independent forward validation horizon.

### The Target for the Full Warehouse Capstone (Weeks 3+)
In the full warehouse release (~79M daily fact rows), we can construct a true **future-window observed target**:
- **Feature window:** Trailing 90 days ($T-90$ to $T$)
- **Target window:** Forward 30 days ($T+1$ to $T+30$)
- **Target definition:** Binary indicator that a page experiences a sustained drop (>= 20%) in impressions or clicks during the forward 30-day window, conditioned on meeting a minimum volume threshold (e.g. >= 100 impressions) to filter out low-volume noise.

### Strict Leakage Guardrail
Because `is_declining_label` in the starter data is computed from `trend_pct` and `trend_direction`, those two columns **must never be used as features** during model training. Feeding `trend_pct` into the feature matrix creates trivial 100% artificial accuracy (target leakage) that collapses in production.

In [2]:
# Code check: Inspect the target proxy distribution and verify volume eligibility
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

print("--- Target Proxy Distribution ---")
print(df['trend_direction'].value_counts())
print(f"\nis_declining_label mean (base rate): {df['is_declining_label'].mean():.4f}")

# Check volume-qualified targets (pages with enough exposure to matter)
visible_declining = df[(df['is_declining_label'] == 1) & (df['impressions_90d'] >= 500)]
print(f"\nPages declining WITH >= 500 impressions: {len(visible_declining):,} pages")
print(f"These at-risk pages account for {visible_declining['impressions_90d'].sum():,} impressions.")


--- Target Proxy Distribution ---
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

is_declining_label mean (base rate): 0.5421

Pages declining WITH >= 500 impressions: 9,961 pages
These at-risk pages account for 79,042,325 impressions.


## 3. Success metric

### One Metric I Can Defend: Precision@K (Specifically Precision@50)

**The primary evaluation metric is Precision@50** (with Precision@20 as secondary review depth).

### Why Precision@K?
1. **Direct Alignment with Human Decision Capacity:** Content teams, SEO strategists, and editors review recommendations in discrete batches — typically 20 to 50 pages per planning cycle. Precision@50 directly asks: *"Of the top 50 pages surfaced by our system to review first, what fraction are genuine high-priority declining/refresh opportunities?"*
2. **Why Generic Accuracy is Misleading:** The dataset base rate is 54.2%. A naive classifier predicting every single page as declining achieves 54.2% accuracy while delivering zero prioritization value. Accuracy rewards correct predictions across thousands of dormant, zero-impact pages in the long tail.
3. **Why ROC-AUC is Secondary:** ROC-AUC measures global ranking across all 30,000 items. While useful to verify the model has learned general signal across all clients, editors never inspect items ranked #15,000 or #25,000. Precision@K measures the exact decision surface where human attention and budget are deployed.

### What Number Means "Good"?
- **Random guess baseline:** 0.542 (the dataset prevalence).
- **Transparent rule baseline:** In the committed starter pipeline (`outputs/model_report.md`), the heuristic scoring rule achieves a Precision@50 of **0.240** (only 12 of the top 50 flagged pages are actually declining, because fixed visibility rules over-index on stale pages that have already flattened out).
- **What "good" means:** Any learned model must substantially outperform the rule baseline on unseen clients. In our starter benchmark, a Random Forest achieves **0.740** Precision@50 (37 of 50 correct) — representing a **3.1x lift** over the heuristic baseline under client-holdout validation. A score of **>= 0.70 Precision@50** under client-holdout split is the threshold for a successful model.

In [3]:
# Code check: Implement precision_at_k and evaluate a simple heuristic score
def precision_at_k(y_true, y_score, k=50):
    """Calculates the precision among the top K highest-scored items."""
    top_k_indices = np.argsort(-y_score)[:k]
    return np.mean(np.array(y_true)[top_k_indices])

y_true = df['is_declining_label'].values

# Baseline heuristic rule: rank purely by volume weighted by age
heuristic_score = df['impressions_90d'].values * (df['content_age_days'].values >= 180)
p50_heuristic = precision_at_k(y_true, heuristic_score, k=50)

print(f"Random baseline Precision@50 (prevalence): {y_true.mean():.3f}")
print(f"Simple age+volume rule Precision@50:       {p50_heuristic:.3f} ({(p50_heuristic*50):.0f}/50 correct)")
print("Reference starter pipeline results (from outputs/model_report.md):")
print("  Baseline rules:  0.240 Precision@50 (12/50 correct)")
print("  Random Forest:   0.740 Precision@50 (37/50 correct) -> Target bar to achieve")


Random baseline Precision@50 (prevalence): 0.542
Simple age+volume rule Precision@50:       0.420 (21/50 correct)
Reference starter pipeline results (from outputs/model_report.md):
  Baseline rules:  0.240 Precision@50 (12/50 correct)
  Random Forest:   0.740 Precision@50 (37/50 correct) -> Target bar to achieve


## 4. The unit of analysis, as a real dataframe

### What is the Unit of Analysis?
**One row = One pseudonymized content page (`content_id`), belonging to a pseudonymized client (`client_id`), observed over a trailing 90-day search and analytics measurement window.**

- **Decision grain:** The content page level. Editors revise articles page by page, not keyword by keyword or day by day.
- **Granularity check:** In this dataset slice, `content_id` is unique (`len(df) == df['content_id'].nunique() == 30,000`).
- **Group structure:** Content items belong to 32 distinct clients. Validation must always use `client_id` grouped holdout so that models are evaluated on unseen client domains rather than memorizing domain-specific quirks.

Below is our lane's slice represented as an actual working dataframe containing the observable feature columns, client groupings, and target proxy.

In [4]:
# Code check: Load and display the unit of analysis as a real dataframe slice
print(f"Verification: df['content_id'].is_unique = {df['content_id'].is_unique}")
print(f"Unique clients: {df['client_id'].nunique()}")

# Select representative columns across identifiers, content features, search signals, and target
key_columns = [
    'content_id',               # Grain key (pseudonym)
    'client_id',                # Grouping key for holdout splits
    'content_type',             # Content category
    'content_age_days',         # Content lifecycle age
    'days_since_last_update',   # Freshness signal
    'word_count',               # Depth measurement
    'impressions_90d',          # Search visibility
    'clicks_90d',               # Search traffic captured
    'avg_position',             # Average SERP position
    'position_tier',            # Position grouping
    'days_with_impressions',    # Impression consistency
    'sessions_90d',             # GA4 onsite visits
    'is_declining_label'        # Target proxy
]

lane_df = df[key_columns].copy()

print("\n--- Real Dataframe Slice (5 sample rows) ---")
display_cols = ['content_id', 'client_id', 'content_type', 'impressions_90d', 'avg_position', 'position_tier', 'is_declining_label']
print(lane_df[display_cols].head(5).to_string(index=False))

print("\n--- Feature Summary Stats ---")
print(lane_df[['content_age_days', 'days_since_last_update', 'word_count', 'impressions_90d', 'avg_position']].describe().round(1))


Verification: df['content_id'].is_unique = True
Unique clients: 32

--- Real Dataframe Slice (5 sample rows) ---
          content_id         client_id    content_type  impressions_90d  avg_position position_tier  is_declining_label
content_304f48230142 client_f369cb89fc keyword article             3803          10.6      striking                   1
content_a1fb4e703a9e client_4e07408562 keyword article            15320          20.3      page_3_5                   1
content_9aa793d4d895 client_7f2253d7e2 keyword article            12581          36.5      page_3_5                   1
content_331d6c4de07b client_19581e27de keyword article            11751           6.2        page_1                   0
content_d99b7a2d90ca client_3fdba35f04 keyword article            19140          44.0      page_3_5                   1

--- Feature Summary Stats ---


       content_age_days  days_since_last_update  word_count  impressions_90d  \
count           30000.0                 30000.0     22301.0          30000.0   
mean              256.2                    46.1      3107.8           5200.4   
std               132.7                    42.1      1452.4          16838.0   
min                90.0                     1.0         8.0              1.0   
25%               132.0                    20.0      2413.0             81.0   
50%               236.0                    20.0      2877.0            731.0   
75%               333.0                   104.0      3666.0           3615.2   
max               564.0                   373.0      9546.0         517715.0   

       avg_position  
count       30000.0  
mean           16.3  
std            15.2  
min             0.0  
25%             6.2  
50%            10.8  
75%            22.3  
max           245.0  


## 5. Why ML beats a fixed rule here

### What Makes the Pattern Too Messy for an If-Statement?

A simple rule-based approach (such as *"if days_since_last_update > 180 and impressions > 500, then refresh"*) fails in real-world search operations for several demonstrable reasons:

1. **Single-Signal Intuitions Fail on Empirical Data:**
   - *Intuition: "Longer articles rank better and stay fresh."* Reality: Median word count for declining pages is 2,909 words; for rising pages, it is 2,848 words. Word count alone has zero discriminative power for content decline.
   - *Intuition: "Target keywords with high search volume guarantee traffic."* Reality: The Pearson correlation between `search_volume` and actual `impressions_90d` is virtually zero ($r \approx 0.001$). A rule built around keyword volume targets phantom traffic.
2. **Combinatorial Explosion of Interacting Signals:**
   Content decay is a multivariate process. A page at position 4 that lost 20% impressions has an entirely different risk profile and operational remedy (SERP competitor displacement -> metadata/snippet audit) compared to a page at position 18 with 5,000 impressions (striking distance opportunity -> content expansion). Hand-crafting nested `if/elif/else` thresholds across 15+ continuous dimensions quickly results in brittle heuristics that overfit historical edge cases.
3. **Non-linear Feature Interactions:**
   The top predictive features identified in the trained model are `days_with_impressions` (consistency, importance 0.158), `log_impressions_90d` (scale, 0.128), `avg_position` (rank, 0.109), and `content_age_days` (lifecycle, 0.096). An ML model (such as a Random Forest or Gradient Boosted Tree) naturally discovers the non-linear interaction boundaries among these signals without manual parameter guessing.
4. **The Empirical Proof:**
   When tested on identical held-out client data, the rule baseline achieved **0.240 Precision@50**, while the Random Forest achieved **0.740 Precision@50**. That is a **3.1x lift in precision** — proving that ML genuinely earns its keep by triaging review capacity far more effectively than fixed rules.

In [5]:
# Code check: Demonstrate empirical failure of simple intuitive rules
print("--- Test Intuition 1: Does word count separate declining from rising content? ---")
down_words = df[df['trend_direction'] == 'down']['word_count'].dropna()
up_words = df[df['trend_direction'] == 'up']['word_count'].dropna()
print(f"Median word count (declining): {down_words.median():.0f} words")
print(f"Median word count (rising):    {up_words.median():.0f} words")
print("-> Word count is identical across both groups. A rule like 'refresh short articles' fails.\n")

print("--- Test Intuition 2: Does keyword search volume correlate with impressions? ---")
valid_vol = df[['search_volume', 'impressions_90d']].dropna()
corr = valid_vol.corr().iloc[0, 1]
print(f"Correlation between search_volume and impressions_90d: {corr:.4f}")
print("-> Correlation is virtually zero. Rules relying on search volume fail.\n")

print("--- Test Intuition 3: What happens with a 'stale visible' if-statement? ---")
stale_rule_matches = df[(df['days_since_last_update'] >= 180) & (df['impressions_90d'] >= 500)]
declining_pages_vis = df[(df['trend_direction'] == 'down') & (df['impressions_90d'] >= 500)]
print(f"Pages matching 'days_since_update >= 180 & impressions >= 500': {len(stale_rule_matches)}")
print(f"Out of {len(declining_pages_vis):,} total declining visible pages")
print(f"Rule coverage: {len(stale_rule_matches) / len(declining_pages_vis):.2%}")
print("-> The fixed rule misses 99.8% of declining visible pages because decay occurs long before 180 days.")


--- Test Intuition 1: Does word count separate declining from rising content? ---


Median word count (declining): 2909 words
Median word count (rising):    2848 words
-> Word count is identical across both groups. A rule like 'refresh short articles' fails.

--- Test Intuition 2: Does keyword search volume correlate with impressions? ---
Correlation between search_volume and impressions_90d: 0.0012
-> Correlation is virtually zero. Rules relying on search volume fail.

--- Test Intuition 3: What happens with a 'stale visible' if-statement? ---
Pages matching 'days_since_update >= 180 & impressions >= 500': 17
Out of 9,961 total declining visible pages
Rule coverage: 0.17%
-> The fixed rule misses 99.8% of declining visible pages because decay occurs long before 180 days.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.